In [ ]:
import pandas as pd
import numpy as np

import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [ ]:

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

train_df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [ ]:
print("=== Info train_df ===")
print(train_df.info())
print("\n=== Info test_df ===")
print(test_df.info())

print("\nTrain shape :", train_df.shape)
print("Test shape  :", test_df.shape)

print("\nValeurs manquantes dans train_df :\n", train_df.isnull().sum())
print("\nValeurs manquantes dans test_df :\n", test_df.isnull().sum())

print("\nDistribution des classes (train_df) :")
print(train_df['Class Index'].value_counts())

=== Info train_df ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Class Index  120000 non-null  int64 
 1   Title        120000 non-null  object
 2   Description  120000 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.7+ MB
None

=== Info test_df ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7600 entries, 0 to 7599
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Class Index  7600 non-null   int64 
 1   Title        7600 non-null   object
 2   Description  7600 non-null   object
dtypes: int64(1), object(2)
memory usage: 178.3+ KB
None

Train shape : (120000, 3)
Test shape  : (7600, 3)

Valeurs manquantes dans train_df :
 Class Index    0
Title          0
Description    0
dtype: int64

Valeurs manquantes dans test_df :
 Class Index    0
Title   

In [ ]:

nlp = spacy.load("en_core_web_sm")


def preprocess(text):
    text = str(text).lower()                      
    text = re.sub(r'[^a-z\s]', '', text)           
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.text not in STOP_WORDS]
    return " ".join(tokens)

train_df['clean_title'] = train_df['Title'].apply(preprocess)
test_df['clean_title'] = test_df['Title'].apply(preprocess)

train_df[['Title', 'clean_title', 'Class Index']].head(5)

,Title,clean_title,Class Index
0,Wall St. Bears Claw Back Into the Black (Reuters),wall st bears claw black reuter,3
1,Carlyle Looks Toward Commercial Aerospace (Reu...,carlyle look commercial aerospace reuter,3
2,Oil and Economy Cloud Stocks' Outlook (Reuters),oil economy cloud stock outlook reuter,3
3,Iraq Halts Oil Exports from Main Southern Pipe...,iraq halt oil export main southern pipeline re...,3
4,"Oil prices soar to all-time record, posing new...",oil price soar alltime record pose new menace ...,3


In [5]:
X_train = train_df['clean_title']
y_train = train_df['Class Index']
X_test = test_df['clean_title']
y_test = test_df['Class Index']

In [6]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),  # unigrams + bigrams
    ('nb', MultinomialNB())
])

In [7]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('nb', MultinomialNB())])

In [8]:
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report :\n", classification_report(y_test, y_pred))

Accuracy: 0.8351315789473684

Classification Report :
               precision    recall  f1-score   support

           1       0.82      0.86      0.84      1900
           2       0.88      0.88      0.88      1900
           3       0.81      0.79      0.80      1900
           4       0.83      0.81      0.82      1900

    accuracy                           0.84      7600
   macro avg       0.84      0.84      0.83      7600
weighted avg       0.84      0.84      0.83      7600



In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [10]:
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('rf', RandomForestClassifier(random_state=42))])

In [11]:
y_pred_rf = rf_pipeline.predict(X_test)
print("Accuracy RF:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report RF:\n", classification_report(y_test, y_pred_rf))

Accuracy RF: 0.8165789473684211

Classification Report RF:
               precision    recall  f1-score   support

           1       0.86      0.83      0.85      1900
           2       0.81      0.87      0.84      1900
           3       0.78      0.79      0.78      1900
           4       0.82      0.78      0.80      1900

    accuracy                           0.82      7600
   macro avg       0.82      0.82      0.82      7600
weighted avg       0.82      0.82      0.82      7600



In [12]:
joblib.dump(pipeline, "news_classifier.pkl")
print("Modèle sauvegardé avec succès !")

Modèle sauvegardé avec succès !


In [19]:
from sklearn.linear_model import LogisticRegression

In [20]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,3))),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

In [21]:
lr_pipeline.fit(train_df['clean_title'], train_df['Class Index'])

c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('lr',
                 LogisticRegression(max_iter=1000, multi_class='multinomial'))])

In [22]:
y_pred_lr = lr_pipeline.predict(test_df['clean_title'])

In [23]:
accuracy = accuracy_score(test_df['Class Index'], y_pred_lr)
report = classification_report(test_df['Class Index'], y_pred_lr)

print("Accuracy LR:", accuracy)
print("\nClassification Report LR:\n", report)

Accuracy LR: 0.8507894736842105

Classification Report LR:
               precision    recall  f1-score   support

           1       0.88      0.86      0.87      1900
           2       0.85      0.92      0.88      1900
           3       0.83      0.81      0.82      1900
           4       0.85      0.82      0.83      1900

    accuracy                           0.85      7600
   macro avg       0.85      0.85      0.85      7600
weighted avg       0.85      0.85      0.85      7600



In [24]:
import joblib

joblib.dump(lr_pipeline, "news_classifier_lr.pkl")
print("Modèle Logistic Regression sauvegardé avec succès !")

Modèle Logistic Regression sauvegardé avec succès !


In [25]:
import joblib
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['clean_title']) 

joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")
print("TF-IDF vectorizer sauvegardé avec succès !")


TF-IDF vectorizer sauvegardé avec succès !
